# Benchmark 2.1 — Part C: Merge & Analisis Final
## Penggabungan Hasil 8 Arsitektur, Tabel Master, Uji McNemar 8×8 & Visualisasi

Notebook ini **menggabungkan hasil evaluasi** dari:
- **Part A** (Google Colab): MobileNetV3Large, MobileNetV2, EfficientNetB0, DenseNet121
- **Part B** (Kaggle): EfficientNetV2S, ConvNeXtTiny, ResNet50, MobileViT-XS

Lalu menghasilkan seluruh tabel metrik saintifik, uji McNemar, visualisasi matriks konfusi, dan grafik *trade-off* deployment untuk naskah riset.

### File yang Dibutuhkan:
Unggah file berikut ke `/content/` atau folder kerja lokal:
- `all_results_partA.pkl`
- `all_results_partB.pkl`
- `deployment_partA.pkl`
- `deployment_partB.pkl`


In [ ]:
# ========================================
# 1. Setup & Pemuatan Artefak Gabungan
# ========================================
import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from scipy import stats as scipy_stats
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 150
sns.set_style("whitegrid")

MERGE_DIR = Path("/content/benchmark2_1_merged") if os.path.exists("/content") else Path("./benchmark2_1_merged")
MERGE_DIR.mkdir(parents=True, exist_ok=True)

# Lokasi file pickle part A dan B
PARTA_RES = Path("/content/all_results_partA.pkl") if os.path.exists("/content/all_results_partA.pkl") else Path("./all_results_partA.pkl")
PARTB_RES = Path("/content/all_results_partB.pkl") if os.path.exists("/content/all_results_partB.pkl") else Path("./all_results_partB.pkl")
PARTA_DEP = Path("/content/deployment_partA.pkl") if os.path.exists("/content/deployment_partA.pkl") else Path("./deployment_partA.pkl")
PARTB_DEP = Path("/content/deployment_partB.pkl") if os.path.exists("/content/deployment_partB.pkl") else Path("./deployment_partB.pkl")

# Jika file belum ada di Colab, beri instruksi upload
if not PARTA_RES.exists() or not PARTB_RES.exists():
    try:
        from google.colab import files
        print("Unggah file all_results_partA.pkl, all_results_partB.pkl, deployment_partA.pkl, deployment_partB.pkl:")
        uploaded = files.upload()
    except Exception as e:
        print(f"Perhatian: Pastikan file pkl diletakkan di path kerja: {e}")

with open(PARTA_RES, "rb") as f:
    res_a = pickle.load(f)
with open(PARTB_RES, "rb") as f:
    res_b = pickle.load(f)
with open(PARTA_DEP, "rb") as f:
    dep_a = pickle.load(f)
with open(PARTB_DEP, "rb") as f:
    dep_b = pickle.load(f)

all_results = {**res_a, **res_b}
deployment  = {**dep_a, **dep_b}

sample_key  = list(all_results.keys())[0]
class_names = all_results[sample_key]["class_names"]
NUM_CLASSES = len(class_names)

MODEL_ORDER = [
    "MobileNetV3Large", "MobileNetV2", "EfficientNetB0", "DenseNet121",
    "EfficientNetV2S", "ConvNeXtTiny", "ResNet50", "MobileViT-XS"
]
MODEL_ORDER = [m for m in MODEL_ORDER if m in all_results]

print(f"✅ Berhasil menggabungkan {len(MODEL_ORDER)} model:")
for m in MODEL_ORDER:
    print(f"  - {m}")
print(f"Kelas ({NUM_CLASSES}): {class_names}")


In [ ]:
# ========================================
# 2. Tabel Master Komparasi 8 Model (Phase 2)
# ========================================
master_rows = []
for name in MODEL_ORDER:
    r   = all_results[name]
    p2  = r["phase2"]
    dep = deployment.get(name, {})
    master_rows.append({
        "Model"          : name,
        "Macro F1"       : f"{np.mean(p2['f1']):.4f} ± {np.std(p2['f1']):.4f}",
        "Accuracy (%)"   : f"{np.mean(p2['acc'])*100:.2f} ± {np.std(p2['acc'])*100:.2f}",
        "Macro Precision": f"{np.mean(p2['prec']):.4f} ± {np.std(p2['prec']):.4f}",
        "Macro Recall"   : f"{np.mean(p2['rec']):.4f} ± {np.std(p2['rec']):.4f}",
        "Params (M)"     : f"{dep.get('params_m', 0):.2f}",
        "TFLite (MB)"    : f"{dep.get('tflite_mb', 0):.2f}",
        "Latency (ms)"   : f"{dep.get('latency_ms', 0):.2f}",
    })

master_df = pd.DataFrame(master_rows)
print("=" * 95)
print("  TABEL MASTER — BENCHMARK 2.1 (8 ARSITEKTUR, 7 KELAS, BALANCED 5-FOLD CV)")
print("=" * 95)
display(master_df) if 'display' in globals() else print(master_df.to_string(index=False))
master_df.to_csv(MERGE_DIR / "benchmark2_1_master_table.csv", index=False)
print("Tersimpan: benchmark2_1_master_table.csv")


In [ ]:
# ========================================
# 3. Metrik Per-Kelas (Precision, Recall, F1)
# ========================================
pc_rows = []
for name in MODEL_ORDER:
    p2 = all_results[name]["phase2"]
    # Rata-rata metrik per kelas dari 5 fold
    p_mean = np.mean(p2["pc_p"], axis=0)
    r_mean = np.mean(p2["pc_r"], axis=0)
    f_mean = np.mean(p2["pc_f1"], axis=0)
    for i, c in enumerate(class_names):
        pc_rows.append({
            "Model"    : name,
            "Kelas"    : c,
            "Precision": round(float(p_mean[i]), 4),
            "Recall"   : round(float(r_mean[i]), 4),
            "F1_Score" : round(float(f_mean[i]), 4),
        })

pc_df = pd.DataFrame(pc_rows)
display(pc_df.head(14)) if 'display' in globals() else print(pc_df.head(14).to_string(index=False))
pc_df.to_csv(MERGE_DIR / "benchmark2_1_per_class_metrics.csv", index=False)
print("Tersimpan: benchmark2_1_per_class_metrics.csv")


In [ ]:
# ========================================
# 4. Uji Signifikansi Statistik McNemar Berpasangan (8×8)
# ========================================
# Gabungkan seluruh prediksi Phase 2 lintas fold
preds_dict = {}
trues_dict = {}
for name in MODEL_ORDER:
    preds_dict[name] = np.concatenate(all_results[name]["phase2"]["y_preds"])
    trues_dict[name] = np.concatenate(all_results[name]["phase2"]["y_trues"])

mcnemar_rows = []
for i in range(len(MODEL_ORDER)):
    for j in range(i + 1, len(MODEL_ORDER)):
        m_a = MODEL_ORDER[i]
        m_b = MODEL_ORDER[j]
        y_true = trues_dict[m_a]
        correct_a = (preds_dict[m_a] == y_true)
        correct_b = (preds_dict[m_b] == y_true)

        # Kontingensi
        b = int(np.sum(correct_a & ~correct_b))  # A benar, B salah
        c = int(np.sum(~correct_a & correct_b))  # A salah, B benar

        if (b + c) == 0:
            p_val = 1.0
        elif (b + c) < 25:
            # Uji Binomial Eksak
            p_val = float(2 * min(scipy_stats.binom.cdf(min(b, c), b + c, 0.5),
                                  1 - scipy_stats.binom.cdf(min(b, c) - 1, b + c, 0.5)))
        else:
            # Uji Chi-Square dengan koreksi kontinuitas Edwards
            chi2 = (abs(b - c) - 1)**2 / (b + c)
            p_val = float(1 - scipy_stats.chi2.cdf(chi2, df=1))
        
        p_val = min(1.0, max(0.0, p_val))
        sig = "Signifikan (p < 0.05)" if p_val < 0.05 else "Tidak Signifikan (Setara)"

        mcnemar_rows.append({
            "Model_A"  : m_a,
            "Model_B"  : m_b,
            "b (A+/B-)": b,
            "c (A-/B+)": c,
            "p_value"  : round(p_val, 4),
            "Kesimpulan": sig,
        })

mcnemar_df = pd.DataFrame(mcnemar_rows)
display(mcnemar_df) if 'display' in globals() else print(mcnemar_df.to_string(index=False))
mcnemar_df.to_csv(MERGE_DIR / "mcnemar_test_results_8x8.csv", index=False)
print("Tersimpan: mcnemar_test_results_8x8.csv")


In [ ]:
# ========================================
# 5. Visualisasi: Confusion Matrix Agregat Seluruh Model
# ========================================
n_models = len(MODEL_ORDER)
cols = 4
rows = math.ceil(n_models / cols)
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))
axes = np.array(axes).reshape(-1)

for idx, name in enumerate(MODEL_ORDER):
    ax = axes[idx]
    y_t = trues_dict[name]
    y_p = preds_dict[name]
    cm  = confusion_matrix(y_t, y_p)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    f1_val = np.mean(all_results[name]["phase2"]["f1"])
    ax.set_title(f"{name}\n(Macro F1: {f1_val:.4f})", fontsize=11, fontweight="bold")
    ax.set_ylabel("True Label", fontsize=9)
    ax.set_xlabel("Predicted Label", fontsize=9)
    ax.tick_params(axis="x", rotation=45)

for idx in range(n_models, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig(MERGE_DIR / "confusion_matrices_7class.png", dpi=200, bbox_inches="tight")
plt.show()
print("Tersimpan: confusion_matrices_7class.png")


In [ ]:
# ========================================
# 6. Visualisasi: Deployment Trade-off Scatter Plot
# ========================================
fig, ax = plt.subplots(figsize=(9, 6))

xs, ys, sizes, labels = [], [], [], []
for name in MODEL_ORDER:
    f1_mean = np.mean(all_results[name]["phase2"]["f1"])
    dep     = deployment.get(name, {})
    lat     = dep.get("latency_ms", 10.0)
    tfl_mb  = dep.get("tflite_mb", 10.0)
    
    xs.append(lat)
    ys.append(f1_mean)
    sizes.append(max(50, tfl_mb * 15))
    labels.append(name)

scatter = ax.scatter(xs, ys, s=sizes, alpha=0.6, edgecolors="black", c=range(len(labels)), cmap="viridis")

for i, txt in enumerate(labels):
    ax.annotate(f"{txt}\n({deployment.get(txt,{}).get('tflite_mb',0):.1f}MB)",
                (xs[i], ys[i]), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=9, fontweight="bold")

ax.set_title("Pareto Trade-off: Macro F1-Score vs CPU Latency vs Ukuran TFLite", fontsize=13, fontweight="bold")
ax.set_xlabel("CPU Latency (ms) [Lebih rendah lebih baik]", fontsize=11)
ax.set_ylabel("Macro F1-Score [Lebih tinggi lebih baik]", fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(MERGE_DIR / "tradeoff_chart_8models.png", dpi=200, bbox_inches="tight")
plt.show()
print("Tersimpan: tradeoff_chart_8models.png")


In [ ]:
# ========================================
# 7. Simpan Gabungan & Arsipkan ke ZIP
# ========================================
with open(MERGE_DIR / "all_results_merged.pkl", "wb") as f:
    pickle.dump(all_results, f)
with open(MERGE_DIR / "deployment_merged.pkl", "wb") as f:
    pickle.dump(deployment, f)

zip_path = shutil.make_archive("/content/benchmark2_1_final_outputs" if os.path.exists("/content") else "./benchmark2_1_final_outputs", "zip", str(MERGE_DIR))
print(f"🎉 SEMUA SELESAI! Hasil akhir diarsipkan: {zip_path}")
